# Merging triangles in STL with decimate from PyVista 0.43/0.45

https://docs.pyvista.org/api/core/_autosummary/pyvista.polydatafilters.decimate

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import numpy as np

import pyvista as pv
#pyvista.__version__
from pyvista.trame.jupyter import launch_server
await launch_server().ready
pv.set_jupyter_backend('html') 
#pv.set_jupyter_backend('trame') 

In [ ]:
stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl'
surf=pv.read(stl_cavity)+pv.read(stl_shell)
surf_shell= pv.read(stl_shell) 

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)
stl_tol=1e-3
stl_tolerance = np.min(spacing) *stl_tol

In [ ]:

'''surf_cavity= surf_cavity.scale(1e-3)   
surf_cavity= surf_cavity.clean(tolerance=1e-7) 
surf_cavity= surf_cavity.fill_holes(1000)  
surf_cavity= surf_cavity.compute_normals(inplace=True, auto_orient_normals=True)   '''

In [ ]:
surf_simple = surf.decimate(0.8, volume_preservation=True,enable_all_attribute_error=True) 

'''
if grid.n_points > 1_000_000:
    print(f"grid is very large ({grid.n_points} points)")
'''

grid.compute_implicit_distance(surf_simple, inplace=True)

'''d_min, d_max = grid['implicit_distance'].min(), grid['implicit_distance'].max()
print(f"Dist range: {d_min:.4f} to {d_max:.4f}")'''

dist = grid["implicit_distance"]
#epsilon = np.mean(grid.spacing) * 0.5
#grid["material_density"] = np.clip(0.5 - (dist / (2 * epsilon)), 0, 1)

pl = pv.Plotter(notebook=True) 
pl.add_mesh(surf_simple, color='cyan', opacity=0.1, style='wireframe', label='Simplified STL')

# debuggin test:
#---------------------------------------------
'''if grid["material_density"].max() >= 0.5:
    inside_voxels = grid.threshold(0.5, scalars='material_density')
    pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
else:
    print("none voxels found inside the threshold, check STL/Grid alignment")
'''
pl.add_legend()
pl.show()